In [ ]:
!pip install -q kagglehub keras-hub
!pip install torch torchvision

In [ ]:
# Imports
import kagglehub
import torch
import torchvision
from torchvision.models.detection import retinanet_resnet50_fpn
from torchvision.models.detection.retinanet import RetinaNetHead
from torchvision import transforms
import torchvision.transforms as T
from torchvision.ops import nms

import xml.etree.ElementTree as ET
from PIL import Image, ImageDraw
import os
import matplotlib.pyplot as plt
import numpy as np
import random
import pandas as pd
from openpyxl import Workbook
from google.colab import files

In [ ]:
# Metrics

def compute_iou(box1, box2):
    x1 = max(box1[0], box2[0])
    y1 = max(box1[1], box2[1])
    x2 = min(box1[2], box2[2])
    y2 = min(box1[3], box2[3])

    inter = max(0, x2 - x1) * max(0, y2 - y1)

    area1 = (box1[2]-box1[0]) * (box1[3]-box1[1])
    area2 = (box2[2]-box2[0]) * (box2[3]-box2[1])

    union = area1 + area2 - inter

    return inter / union if union > 0 else 0


def evaluate_image(pred_boxes, pred_scores, gt_boxes, iou_thresh=0.5, score_thresh=0.15):
    pred_boxes = pred_boxes[pred_scores > score_thresh]

    matched = set()
    tp = 0
    fp = 0

    for pb in pred_boxes:
        found_match = False
        for i, gb in enumerate(gt_boxes):
            if i in matched:
                continue

            iou = compute_iou(pb, gb)

            if iou >= iou_thresh:
                tp += 1
                matched.add(i)
                found_match = True
                break

        if not found_match:
            fp += 1

    fn = len(gt_boxes) - len(matched)

    return tp, fp, fn


def evaluate_dataset(model, data, device):
    model_was_training = model.training  # save state
    model.eval()

    total_tp, total_fp, total_fn = 0, 0, 0

    with torch.no_grad():
        for image, target in data:
            image = image.to(device)
            preds = model([image])[0]

            pred_boxes = preds["boxes"].cpu()
            pred_scores = preds["scores"].cpu()
            gt_boxes = target["boxes"]

            tp, fp, fn = evaluate_image(pred_boxes, pred_scores, gt_boxes)

            total_tp += tp
            total_fp += fp
            total_fn += fn

    # restore mode
    if model_was_training:
        model.train()

    precision = total_tp / (total_tp + total_fp + 1e-6)
    recall = total_tp / (total_tp + total_fn + 1e-6)

    return precision, recall


def compute_map50(model, data, device, iou_thresh=0.5, score_thresh=0.15):
    model.eval()

    all_detections = []
    total_gt = 0

    with torch.no_grad():
        for image, target in data:
            image = image.to(device)
            preds = model([image])[0]

            pred_boxes = preds["boxes"].cpu()
            pred_scores = preds["scores"].cpu()

            gt_boxes = target["boxes"]
            total_gt += len(gt_boxes)

            matched = set()

            # sort predictions by confidence
            sorted_indices = torch.argsort(pred_scores, descending=True)

            for idx in sorted_indices:
                score = pred_scores[idx]

                if score < score_thresh:
                    continue

                pb = pred_boxes[idx]

                best_iou = 0
                best_gt_idx = -1

                for i, gb in enumerate(gt_boxes):
                    if i in matched:
                        continue

                    iou = compute_iou(pb, gb)

                    if iou > best_iou:
                        best_iou = iou
                        best_gt_idx = i

                if best_iou >= iou_thresh:
                    all_detections.append((score.item(), 1))  # TP
                    matched.add(best_gt_idx)
                else:
                    all_detections.append((score.item(), 0))  # FP

    # sort all detections globally
    all_detections.sort(key=lambda x: x[0], reverse=True)

    tp_cumsum = 0
    fp_cumsum = 0

    precisions = []
    recalls = []

    for _, is_tp in all_detections:
        if is_tp:
            tp_cumsum += 1
        else:
            fp_cumsum += 1

        precision = tp_cumsum / (tp_cumsum + fp_cumsum + 1e-6)
        recall = tp_cumsum / (total_gt + 1e-6)

        precisions.append(precision)
        recalls.append(recall)

    # convert to numpy
    precisions = np.array(precisions)
    recalls = np.array(recalls)

    # compute AP (area under curve)
    ap = 0.0
    for i in range(1, len(recalls)):
        ap += (recalls[i] - recalls[i-1]) * precisions[i]

    return ap

In [ ]:
# Download dataset from Kaggle

downloaded_data_dir = kagglehub.dataset_download("andrewmvd/car-plate-detection")
print("Downloaded to:", downloaded_data_dir)

# Folder paths

images_dir = os.path.join(downloaded_data_dir, "images")
annotations_dir = os.path.join(downloaded_data_dir, "annotations")

print("Images dir:", images_dir)
print("Annotations dir:", annotations_dir)

print("Number of image files:", len(os.listdir(images_dir)))
print("Number of annotation files:", len(os.listdir(annotations_dir)))

print(os.listdir(images_dir)[:5])
print(os.listdir(annotations_dir)[:5])

In [ ]:
# Parse xml file

def parse_xml(xml_file):
    tree = ET.parse(xml_file)
    root = tree.getroot()

    boxes = []
    labels = []

    for obj in root.findall("object"):
        xmin = int(obj.find("bndbox/xmin").text)
        ymin = int(obj.find("bndbox/ymin").text)
        xmax = int(obj.find("bndbox/xmax").text)
        ymax = int(obj.find("bndbox/ymax").text)

        if xmax > xmin and ymax > ymin:
            boxes.append([xmin, ymin, xmax, ymax])
            labels.append(1)

    return boxes, labels

#Data Augmentation

In [ ]:
# Add noise to simulate low-light cameras

class AddNoise:
    def __call__(self, img):
        noise = torch.randn_like(img) * 0.05
        return img + noise

In [ ]:
# Data Augmentation Pipeline

SIZE = 300

transform = T.Compose([
    T.Resize((SIZE, SIZE)),

     # Night simulation (ONLY sometimes)
    T.RandomApply([
        T.ColorJitter(
            brightness=0.75,
        )
    ], p=100),

    # Blur (ONLY sometimes)
    T.RandomApply([
        T.GaussianBlur(kernel_size=3, sigma=(0.6, 15))
    ], p=0.4),

    T.ToTensor(),
])

#Load Data

In [ ]:
# Load dataset

data = []
image_files = os.listdir(images_dir)

for img_name in image_files:
    img_path = os.path.join(images_dir, img_name)
    xml_path = os.path.join(annotations_dir, img_name.replace(".png", ".xml"))

    if not os.path.exists(xml_path):
        continue

    image = Image.open(img_path).convert("RGB")
    image = transform(image)

    boxes, labels = parse_xml(xml_path)

    if len(boxes) == 0:
        continue

    target = {
        "boxes": torch.tensor(boxes, dtype=torch.float32),
        "labels": torch.tensor(labels, dtype=torch.int64)
    }

    data.append((image, target))

print("Total samples:", len(data))

In [ ]:
# Shuffle and split (80/20)

random.shuffle(data)

split_idx = int(0.8 * len(data))

train_data = data[:split_idx]
val_data = data[split_idx:]

print("Train samples:", len(train_data))
print("Validation samples:", len(val_data))

In [ ]:
# Load image

img_path = os.path.join(images_dir, image_files[0])  # or your custom image
image = Image.open(img_path).convert("RGB")

input_tensor = transform(image)

# Number of augmented samples
NUM_SAMPLES = 10

plt.figure(figsize=(15, 5))

for i in range(NUM_SAMPLES):
    augmented = transform(image)

    # convert tensor to image
    aug_img = augmented.permute(1, 2, 0).numpy()

    plt.subplot(1, NUM_SAMPLES, i + 1)
    plt.imshow(aug_img)
    plt.axis("off")
    plt.title(f"Aug {i+1}")

plt.suptitle("Augmented Samples", fontsize=16)
plt.show()

In [ ]:
# Batch generator

def get_batch(data, batch_size):
    for i in range(0, len(data), batch_size):
        batch = data[i:i+batch_size]
        images = [item[0] for item in batch]
        targets = [item[1] for item in batch]
        yield images, targets

In [ ]:
# Load model

model = retinanet_resnet50_fpn(weights="DEFAULT")

# replace head
num_classes = 2

in_channels = model.backbone.out_channels
num_anchors = model.head.classification_head.num_anchors

model.head = RetinaNetHead(in_channels, num_anchors, num_classes)


for param in model.backbone.parameters():
    param.requires_grad = True

# device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

In [ ]:
# Recreate model (EXACT SAME architecture) - LOAD MODEL (WEIGHTS)
model = retinanet_resnet50_fpn(weights="DEFAULT")

num_classes = 2
in_channels = model.backbone.out_channels
num_anchors = model.head.classification_head.num_anchors

model.head = RetinaNetHead(in_channels, num_anchors, num_classes)

# Load weights
checkpoint = torch.load("/content/drive/MyDrive/Copy of checkpoint_augmented_new3_35_1e-6.pth")

model.load_state_dict(checkpoint['model_state'])

# device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

# switch to training mode
model.train()

In [ ]:
for name, param in model.backbone.named_parameters():
    if "layer3" in name or "layer4" in name:
        param.requires_grad = True
    else:
        param.requires_grad = False

In [ ]:
# Unfreeze backbone
for param in model.backbone.parameters():
    param.requires_grad = True

# very low LR
optimizer = torch.optim.Adam(model.parameters(), lr=5e-6)

In [ ]:
# Metrics history
loss_history = []

precision_train_history = []
recall_train_history = []
map50_train_history = []

_precision_val_history = []
recall_val_history = []
map50_val_history = []

scaler = torch.amp.GradScaler("cuda")

In [ ]:
# Hyper-parameters
EPOCHS = 20
BATCH_SIZE = 2

In [ ]:
# Optimizer
optimizer = torch.optim.Adam(model.parameters(), lr=1e-6)

In [ ]:
# Training and Validation Loops

model.train()

for epoch in range(EPOCHS):
    total_loss = 0

    # Training loop
    model.train()

    for images, targets in get_batch(train_data, BATCH_SIZE):
        images = [img.to(device) for img in images]
        targets = [{k: v.to(device) for k, v in t.items()} for t in targets]

        with torch.amp.autocast("cuda"):
            loss_dict = model(images, targets)
            loss = sum(loss_dict.values())

        optimizer.zero_grad()
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        total_loss += loss.item()

    # Validation Loop
    model.eval()

    with torch.no_grad():
        # Train metrics
        train_precision, train_recall = evaluate_dataset(model, train_data, device)
        train_map50 = compute_map50(model, train_data, device)

        # Val metrics
        val_precision, val_recall = evaluate_dataset(model, val_data, device)
        val_map50 = compute_map50(model, val_data, device)


    # Saving metrics history
    loss_history.append(total_loss)

    precision_train_history.append(train_precision)
    recall_train_history.append(train_recall)
    map50_train_history.append(train_map50)

    precision_val_history.append(val_precision)
    recall_val_history.append(val_recall)
    map50_val_history.append(val_map50)


    print(
        f"Epoch {epoch+1}, Loss: {total_loss:.4f}, "
        f"Train mAP50: {train_map50:.4f}, "
        f"Val mAP50: {val_map50:.4f}"
    )

    # Plot
    plt.figure(figsize=(10,5))

    plt.plot(map50_train_history, label="Train mAP@50")
    plt.plot(map50_val_history, label="Val mAP@50")

    plt.plot(precision_train_history, linestyle='--', label="Train Precision")
    plt.plot(precision_val_history, linestyle='--', label="Val Precision")

    plt.plot(recall_train_history, linestyle=':', label="Train Recall")
    plt.plot(recall_val_history, linestyle=':', label="Val Recall")

    plt.xlabel("Epoch")
    plt.ylabel("Value")
    plt.title("Train vs Validation Metrics")
    plt.legend()
    plt.grid()

    plt.show()

    model.train()

#Saving Metrics History And Model Weights

In [ ]:
# Create workbook
wb = Workbook()
ws = wb.active
ws.title = "Validation Metrics"

# Headers
ws.append(["Epoch", "Loss", "Precision", "Recall", "mAP50"])

# Fill data
for i in range(len(loss_history)):
    ws.append([
        i + 1,
        loss_val_history[i],
        precision_val_history[i],
        recall_val_history[i],
        map50_val_history[i]
    ])

# Save file
file_name = "retinanet_augmented_darker+blurrier_training_metrics.xlsx"
wb.save(file_name)

print(f"Saved to {file_name}")

In [ ]:
# Save model
torch.save({
    'model_state': model.state_dict(),
    'optimizer_state': optimizer.state_dict(),
}, "checkpoint_augmented_darker+stronger_90_1e-6.pth")

In [ ]:
# Download metrics sheet and model weights
files.download("retinanet_augmented_darker+blurrier_training_metrics.xlsx")
files.download("checkpoint_augmented_darker+stronger_70_1e-6.pth")

#Model Evaluation - Testing on Images

In [ ]:
# Evaluation mode
model.eval()


# Load image
img_path = os.path.join(images_dir, image_files[208])  # or your custom image
image = Image.open(img_path).convert("RGB")

input_tensor = transform(image).to(device)

# Predict
with torch.no_grad():
    preds = model([input_tensor])[0]

if len(preds["scores"]) <= 0:
    print("No predictions returned")

# Prepare drawing
draw = ImageDraw.Draw(image)

boxes = preds["boxes"].cpu()
scores = preds["scores"].cpu()

# Apply NMS
if len(boxes) > 0:
    keep = nms(boxes, scores, iou_threshold=0.5)
    boxes = boxes[keep]
    scores = scores[keep]

# Sort by confidence
sorted_idx = torch.argsort(scores, descending=True)
boxes = boxes[sorted_idx]
scores = scores[sorted_idx]

# Top k detections
TOP_K = 1
boxes = boxes[:TOP_K]
scores = scores[:TOP_K]

# Threshold
threshold = 0.5

# Scale boxes back to original image size
orig_w, orig_h = image.size

for box, score in zip(boxes, scores):
    if score > threshold:
        x1, y1, x2, y2 = map(int, box)

        draw.rectangle([x1, y1, x2, y2], outline="red", width=3)
        draw.text((x1, y1), f"{score:.2f}", fill="red")

# Result
plt.imshow(image)
plt.axis("off")
plt.show()

In [ ]:
# Evaluation mode
model.eval()

# Load image
img_path = "/content/car_nigh_5.jfif"   # 👈 change this
image = Image.open(img_path).convert("RGB")

transform = T.Compose([
    T.Resize((300, 300)),
    T.ToTensor(),
])

input_tensor = transform(image).to(device)

# Predict
with torch.no_grad():
    preds = model([input_tensor])[0]

# Debug
print("Num boxes:", len(preds["boxes"]))
if len(preds["scores"]) > 0:
    print("Max score:", preds["scores"].max().item())
    print("Top scores:", preds["scores"][:10])
else:
    print("No predictions returned")

# Prepare drawing
draw = ImageDraw.Draw(image)

boxes = preds["boxes"].cpu()
scores = preds["scores"].cpu()

# Apply NMS
if len(boxes) > 0:
    keep = nms(boxes, scores, iou_threshold=0.5)
    boxes = boxes[keep]
    scores = scores[keep]

# Sort by confidence
sorted_idx = torch.argsort(scores, descending=True)
boxes = boxes[sorted_idx]
scores = scores[sorted_idx]

# Top k detections
TOP_K = 1
boxes = boxes[:TOP_K]
scores = scores[:TOP_K]

threshold = 0.5

# Scale boxes back to original image size
orig_w, orig_h = image.size

for box, score in zip(boxes, scores):
    if score > threshold:
        x1, y1, x2, y2 = box

        x1 = int(x1 * orig_w / 300)
        x2 = int(x2 * orig_w / 300)
        y1 = int(y1 * orig_h / 300)
        y2 = int(y2 * orig_h / 300)

        draw.rectangle([x1, y1, x2, y2], outline="red", width=3)
        draw.text((x1, y1), f"{score:.2f}", fill="red")

# Show result
plt.imshow(image)
plt.axis("off")
plt.show()